In [0]:
df = spark.table("`cafe_sales_2026`.bronze.dirty_cafe_sales")
display(df)

In [0]:
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull() | (col(c) == "UNKNOWN") | (col(c) == "ERROR"), c)).alias(c)
    for c in df.columns
]).display()

In [0]:
total_linhas = df.count()
print(f"Total de linhas: {total_linhas}")

In [0]:
from pyspark.sql.functions import col, when, round as spark_round, expr

df_clean = df

# 1. Tratar valores nulos/inválidos em Payment_Method e Location -> "não informado"
df_clean = df_clean.withColumn(
    "Payment_Method",
    when(col("Payment_Method").isNull() | (col("Payment_Method").isin("UNKNOWN", "ERROR")), "não informado")
    .otherwise(col("Payment_Method"))
)

df_clean = df_clean.withColumn(
    "Location",
    when(col("Location").isNull() | (col("Location").isin("UNKNOWN", "ERROR")), "não informado")
    .otherwise(col("Location"))
)

# 2. Tratar Item -> "não informado" (não há como recalcular)
df_clean = df_clean.withColumn(
    "Item",
    when(col("Item").isNull() | (col("Item").isin("UNKNOWN", "ERROR")), "não informado")
    .otherwise(col("Item"))
)

# 3. Converter colunas numéricas usando try_cast (valores inválidos viram null)
df_clean = df_clean.withColumn("Quantity", expr("try_cast(Quantity AS INT)"))
df_clean = df_clean.withColumn("Price_Per_Unit", expr("try_cast(Price_Per_Unit AS DOUBLE)"))
df_clean = df_clean.withColumn("Total_Spent", expr("try_cast(Total_Spent AS DOUBLE)"))

# 4. Recalcular Total_Spent quando possível (Quantity x Price_Per_Unit)
df_clean = df_clean.withColumn(
    "Total_Spent",
    when(col("Total_Spent").isNull() & col("Quantity").isNotNull() & col("Price_Per_Unit").isNotNull(),
         spark_round(col("Quantity") * col("Price_Per_Unit"), 2))
    .otherwise(col("Total_Spent"))
)

# 5. Recalcular Price_Per_Unit quando possível
df_clean = df_clean.withColumn(
    "Price_Per_Unit",
    when(col("Price_Per_Unit").isNull() & col("Quantity").isNotNull() & col("Total_Spent").isNotNull() & (col("Quantity") != 0),
         spark_round(col("Total_Spent") / col("Quantity"), 2))
    .otherwise(col("Price_Per_Unit"))
)

# 6. Recalcular Quantity quando possível
df_clean = df_clean.withColumn(
    "Quantity",
    when(col("Quantity").isNull() & col("Total_Spent").isNotNull() & col("Price_Per_Unit").isNotNull() & (col("Price_Per_Unit") != 0),
         spark_round(col("Total_Spent") / col("Price_Per_Unit"), 0).cast("int"))
    .otherwise(col("Quantity"))
)

# 7. Tratar Transaction_Date -> "não informado" (não há como recalcular)
df_clean = df_clean.withColumn(
    "Transaction_Date",
    when(col("Transaction_Date").isNull() | (col("Transaction_Date").isin("UNKNOWN", "ERROR")), "Não Informado")
    .otherwise(col("Transaction_Date"))
)

display(df_clean)
print(f"Linhas totais: {df_clean.count()} de {total_linhas} originais")

In [0]:
df_clean.write.mode("overwrite").saveAsTable("cafe_sales_2026.silver.clean_cafe_sales")